# Hera Quickstart (Two Paths) + Extras: Additional Example & Cleanup Operations
This notebook extends the previous **two-paths** quickstart with:

1) An additional example using **`LowFreqToolkit`** with a **Parquet** data source (`YAVNEEL_PARQUET`).
2) **Cleanup operations**:
   - Unbind a data source from a toolkit.
   - Delete a data source from the repository (with optional cascade removal from all bindings).


## Prerequisites
- Local Hera repo (e.g., `~/hera/hera`) and active virtualenv (e.g., `heraenv`).
- If using the CLI path:
```bash
cd ~/hera/hera
export PATH="$PWD/bin:$PATH"
export PYTHONPATH="$PWD:$PYTHONPATH"
hera-project --help
```
**Pick either Path A or Path B to create the project. Then continue to Sections 3–7.**


## Path A — WITH CLI (create project)
This path uses the existing CLI and silences a known stderr message so the notebook doesn't fail.


In [1]:
%%bash
cd ~/hera/hera/tests
~/hera/hera/bin/hera-project project create DEMO --directory . 2>/dev/null || true
~/hera/hera/bin/hera-project project list --onlyName 2>/dev/null || true
ls -la DEMO || true


DEBUG   : hera-project/<module>(138) Got Namespace(projectName='DEMO', directory='.', loadRepositories=True, overwrite=True, func=<function project_create at 0x7b546b4e31f0>) in the command line
Created project DEMO in directory .
INFO    : project.py/__init__(221) Initializing with logger defaultProject
DEBUG   : hera-project/<module>(138) Got Namespace(connectionName=None, fulldetails=False, func=<function project_list at 0x793cdc30fca0>) in the command line
DEBUG   : jsonutils.py/ConfigurationToJSON(75) Processing {}


Projects in the connection ilay
-------------------------------
                    Project Name                 
0                                                
30                                   DEMO_PROJECT
1                                   DOCUMENTATION
26                                     Demography
12                      DynamicToolkitTestProject
10                                            IMS
20                                  Jerusalem2018
28      

## Path B — NO CLI (create minimal skeleton)
This path builds `DEMO/` with minimal repository JSON files directly.


In [3]:
# Python: ensure minimal project skeleton at ~/hera/hera/tests/DEMO
from pathlib import Path
import json

root = Path('~/hera/hera/tests/DEMO').expanduser()
for p in [root/'repositories', root/'data', root/'outputs']:
    p.mkdir(parents=True, exist_ok=True)

repo = root/'repositories'
defaults = {
    'datasources.json': [],
    'toolkits.json': [],
    'bindings.json': {}
}
for name, default in defaults.items():
    f = repo/name
    if not f.exists():
        f.write_text(json.dumps(default, ensure_ascii=False, indent=2), encoding='utf-8')

print('Project skeleton is ready at:', root)


Project skeleton is ready at: /home/ilay/hera/hera/tests/DEMO


## 3) Upsert Data Sources (shared for both paths)
Add or update two data sources into `DEMO/repositories/datasources.json`: one HGT and one CSV.
No real files are required; paths may be placeholders.


In [4]:
# Upsert SRTMGL1_TILE (HGT)
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
dsfile = root / 'repositories' / 'datasources.json'
arr = json.loads(dsfile.read_text(encoding='utf-8')) if dsfile.exists() else []
item = {'name':'SRTMGL1_TILE','type':'elevation/hgt','path':'data/topography/N33E035.SRTMGL1.hgt','format':'HGT','metadata':{'tile':'N33E035','crs':'EPSG:4326'}}
for i,d in enumerate(arr):
    if d.get('name')==item['name']:
        arr[i]=item; break
else:
    arr.append(item)
dsfile.parent.mkdir(parents=True, exist_ok=True)
dsfile.write_text(json.dumps(arr, ensure_ascii=False, indent=2), encoding='utf-8')
print('OK wrote', dsfile)


OK wrote /home/ilay/hera/hera/tests/DEMO/repositories/datasources.json


In [5]:
# Upsert EXTRA_DS (CSV)
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
dsfile = root / 'repositories' / 'datasources.json'
arr = json.loads(dsfile.read_text(encoding='utf-8')) if dsfile.exists() else []
item = {'name':'EXTRA_DS','type':'table/csv','path':'data/my.csv','format':'csv','metadata':{}}
for i, d in enumerate(arr):
    if d.get('name')==item['name']:
        arr[i]=item; break
else:
    arr.append(item)
dsfile.parent.mkdir(parents=True, exist_ok=True)
dsfile.write_text(json.dumps(arr, ensure_ascii=False, indent=2), encoding='utf-8')
print('OK wrote', dsfile)


OK wrote /home/ilay/hera/hera/tests/DEMO/repositories/datasources.json


## 4) Bind those data sources to `TopographyToolkit` (shared)
Store the mapping in `DEMO/repositories/bindings.json`.


In [6]:
# Bind SRTMGL1_TILE and EXTRA_DS to TopographyToolkit
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
bfile = root / 'repositories' / 'bindings.json'
bindings = json.loads(bfile.read_text(encoding='utf-8')) if bfile.exists() else {}
tk='TopographyToolkit'
for ds in ('SRTMGL1_TILE','EXTRA_DS'):
    lst = list(bindings.get(tk, []))
    if ds not in lst:
        lst.append(ds)
    bindings[tk] = lst
bfile.write_text(json.dumps(bindings, ensure_ascii=False, indent=2), encoding='utf-8')
print('OK wrote', bfile)


OK wrote /home/ilay/hera/hera/tests/DEMO/repositories/bindings.json


### Verify (TopographyToolkit)


In [7]:
# Show current bindings for TopographyToolkit
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
bindings = json.loads((root/'repositories'/'bindings.json').read_text(encoding='utf-8'))
datasources = json.loads((root/'repositories'/'datasources.json').read_text(encoding='utf-8'))
by_name = {d['name']: d for d in datasources}
print('Data sources for TopographyToolkit:')
for name in bindings.get('TopographyToolkit', []):
    d = by_name.get(name, {'name': name})
    print(f"- {d.get('name')} | type={d.get('type')} | path={d.get('path')} | format={d.get('format')}")


Data sources for TopographyToolkit:
- SRTMGL1_TILE | type=elevation/hgt | path=data/topography/N33E035.SRTMGL1.hgt | format=HGT
- EXTRA_DS | type=table/csv | path=data/my.csv | format=csv


## 5) Additional example: `LowFreqToolkit` + `YAVNEEL_PARQUET` (Parquet)
Register a Parquet data source and bind it to `LowFreqToolkit`.


In [8]:
# Upsert YAVNEEL_PARQUET (Parquet)
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
dsfile = root / 'repositories' / 'datasources.json'
arr = json.loads(dsfile.read_text(encoding='utf-8')) if dsfile.exists() else []
item = {'name':'YAVNEEL_PARQUET','type':'meteorology/lowfreq','path':'data/meteorology/lowfreq/YAVNEEL.parquet','format':'parquet','metadata':{'station':'Yavneel'}}
for i,d in enumerate(arr):
    if d.get('name')==item['name']:
        arr[i]=item; break
else:
    arr.append(item)
dsfile.parent.mkdir(parents=True, exist_ok=True)
dsfile.write_text(json.dumps(arr, ensure_ascii=False, indent=2), encoding='utf-8')
print('OK wrote', dsfile)


OK wrote /home/ilay/hera/hera/tests/DEMO/repositories/datasources.json


In [9]:
# Bind YAVNEEL_PARQUET to LowFreqToolkit
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
bfile = root / 'repositories' / 'bindings.json'
bindings = json.loads(bfile.read_text(encoding='utf-8')) if bfile.exists() else {}
tk='LowFreqToolkit'
lst = list(bindings.get(tk, []))
if 'YAVNEEL_PARQUET' not in lst:
    lst.append('YAVNEEL_PARQUET')
bindings[tk] = lst
bfile.write_text(json.dumps(bindings, ensure_ascii=False, indent=2), encoding='utf-8')
print('OK wrote', bfile)


OK wrote /home/ilay/hera/hera/tests/DEMO/repositories/bindings.json


### Verify (LowFreqToolkit)


In [10]:
# Show current bindings for LowFreqToolkit
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
bindings = json.loads((root/'repositories'/'bindings.json').read_text(encoding='utf-8'))
datasources = json.loads((root/'repositories'/'datasources.json').read_text(encoding='utf-8'))
by_name = {d['name']: d for d in datasources}
print('Data sources for LowFreqToolkit:')
for name in bindings.get('LowFreqToolkit', []):
    d = by_name.get(name, {'name': name})
    print(f"- {d.get('name')} | type={d.get('type')} | path={d.get('path')} | format={d.get('format')}")


Data sources for LowFreqToolkit:
- YAVNEEL_PARQUET | type=meteorology/lowfreq | path=data/meteorology/lowfreq/YAVNEEL.parquet | format=parquet


## 6) Cleanup: Unbind a data source from a toolkit
This removes a single data source from a toolkit's binding list, without deleting the data source from the repository.


In [11]:
# Unbind 'EXTRA_DS' from 'TopographyToolkit' (example)
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
bfile = root / 'repositories' / 'bindings.json'
bindings = json.loads(bfile.read_text(encoding='utf-8')) if bfile.exists() else {}
tk = 'TopographyToolkit'
remove_ds = 'EXTRA_DS'
lst = [x for x in bindings.get(tk, []) if x != remove_ds]
bindings[tk] = lst
bfile.write_text(json.dumps(bindings, ensure_ascii=False, indent=2), encoding='utf-8')
print('Unbound', remove_ds, 'from', tk)


Unbound EXTRA_DS from TopographyToolkit


### Verify after unbind


In [12]:
# Show bindings for TopographyToolkit after unbind
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
bindings = json.loads((root/'repositories'/'bindings.json').read_text(encoding='utf-8'))
datasources = json.loads((root/'repositories'/'datasources.json').read_text(encoding='utf-8'))
by_name = {d['name']: d for d in datasources}
print('Data sources for TopographyToolkit:')
for name in bindings.get('TopographyToolkit', []):
    d = by_name.get(name, {'name': name})
    print(f"- {d.get('name')} | type={d.get('type')} | path={d.get('path')} | format={d.get('format')}")


Data sources for TopographyToolkit:
- SRTMGL1_TILE | type=elevation/hgt | path=data/topography/N33E035.SRTMGL1.hgt | format=HGT


## 7) Cleanup: Delete a data source (with optional cascade from all bindings)
Deleting a data source means removing its entry from `datasources.json`. **Optional cascade**: also remove its name from *all* toolkit bindings.

Below we delete `EXTRA_DS` from the repository and cascade-remove it from every toolkit's binding list.


In [13]:
# Delete 'EXTRA_DS' from datasources.json and cascade-remove from all bindings
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
dsfile = root / 'repositories' / 'datasources.json'
bfile = root / 'repositories' / 'bindings.json'

# 1) Remove from datasources.json
arr = json.loads(dsfile.read_text(encoding='utf-8')) if dsfile.exists() else []
arr = [d for d in arr if d.get('name') != 'EXTRA_DS']
dsfile.write_text(json.dumps(arr, ensure_ascii=False, indent=2), encoding='utf-8')

# 2) Cascade-remove from bindings.json
bindings = json.loads(bfile.read_text(encoding='utf-8')) if bfile.exists() else {}
for tk, lst in list(bindings.items()):
    bindings[tk] = [x for x in lst if x != 'EXTRA_DS']
bfile.write_text(json.dumps(bindings, ensure_ascii=False, indent=2), encoding='utf-8')

print('Deleted EXTRA_DS and cascaded removal from all bindings.')


Deleted EXTRA_DS and cascaded removal from all bindings.


### Verify after delete


In [14]:
All instructions are in English.
# Show two toolkits to confirm cleanup
import json, pathlib
root = pathlib.Path('~/hera/hera/tests/DEMO').expanduser()
bindings = json.loads((root/'repositories'/'bindings.json').read_text(encoding='utf-8'))
datasources = json.loads((root/'repositories'/'datasources.json').read_text(encoding='utf-8'))
by_name = {d['name']: d for d in datasources}
for tk in ('TopographyToolkit','LowFreqToolkit'):
    print(f"\nData sources for {tk}:")
    for name in bindings.get(tk, []):
        d = by_name.get(name, {'name': name})
        print(f"- {d.get('name')} | type={d.get('type')} | path={d.get('path')} | format={d.get('format')}")



Data sources for TopographyToolkit:
- SRTMGL1_TILE | type=elevation/hgt | path=data/topography/N33E035.SRTMGL1.hgt | format=HGT

Data sources for LowFreqToolkit:
- YAVNEEL_PARQUET | type=meteorology/lowfreq | path=data/meteorology/lowfreq/YAVNEEL.parquet | format=parquet
